In [ ]:
!pip install -qU \
    fastapi \
    uvicorn \
    python-dotenv \
    openai==1.23.6 \
    langchain \
    langchain-openai==0.0.8 \
    langchain-community \
    faiss-cpu \
    requests \
    nest-asyncio \
    httpx

In [ ]:

import sys
import openai
import fastapi
import langchain

print("Python:", sys.version)
print("OpenAI:", openai.__version__)
print("FastAPI:", fastapi.__version__)
print("LangChain:", langchain.__version__)

print("\nAll packages loaded successfully.")

In [3]:

import os
from getpass import getpass

api_key = getpass("Enter your OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = api_key

print("API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

Enter your OpenAI API key: ··········
API key loaded: True


In [4]:

%%writefile day30-ai-research-assistant/backend/requirements.txt

fastapi
uvicorn[standard]
python-dotenv
openai
pydantic
requests
langchain
langchain-openai
langchain-community
faiss-cpu

Overwriting day30-ai-research-assistant/backend/requirements.txt


In [5]:

%%writefile day30-ai-research-assistant/backend/.env.example

OPENAI_API_KEY=your_openai_api_key_here

Overwriting day30-ai-research-assistant/backend/.env.example


In [6]:

import os

key = os.environ.get("OPENAI_API_KEY")

with open(
    "day30-ai-research-assistant/backend/.env",
    "w"
) as f:
    f.write(
        f"OPENAI_API_KEY={key}\n"
    )

print(".env created.")

.env created.


In [7]:

%%writefile day30-ai-research-assistant/backend/research.py

import os
import requests

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY is not configured."
    )


llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,
    api_key=OPENAI_API_KEY
)


embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY
)


def progress(callback, step, message):

    if callback:
        callback(step, message)


def get_wikipedia_information(topic):

    url = (
        "https://en.wikipedia.org/api/rest_v1/"
        "page/summary/"
        + requests.utils.quote(topic)
    )

    try:

        response = requests.get(
            url,
            timeout=20,
            headers={
                "User-Agent":
                "Day30-AI-Research-Assistant/1.0"
            }
        )

        if response.status_code == 200:

            data = response.json()

            return {
                "title": data.get(
                    "title",
                    topic
                ),
                "text": data.get(
                    "extract",
                    ""
                ),
                "url": data.get(
                    "content_urls",
                    {})
                    .get("desktop", {})
                    .get("page", "")
            }

    except Exception:
        pass

    return {
        "title": topic,
        "text": "",
        "url": ""
    }


def create_faiss_store(topic, information):

    documents = []

    if information["text"]:

        documents.append(
            Document(
                page_content=information["text"],
                metadata={
                    "source":
                    information["url"]
                }
            )
        )


    documents.append(
        Document(
            page_content=f"""
Research topic:
{topic}

The research assistant should analyze this topic
from multiple perspectives including benefits,
risks, challenges, applications, limitations,
and future implications.
""",
            metadata={
                "source": "research-planning"
            }
        )
    )


    vector_store = FAISS.from_documents(
        documents,
        embeddings
    )

    return vector_store


def generate_research(
    topic,
    progress_callback=None
):

    # ==========================================
    # STEP 1
    # ==========================================

    progress(
        progress_callback,
        1,
        "Understanding the research topic..."
    )


    planning_prompt = f"""
You are an expert research planner.

Research topic:
{topic}

Create a research plan.

Identify:

1. Main research question
2. Important subtopics
3. Key concepts
4. Benefits to investigate
5. Risks to investigate
6. Real-world applications
7. Future developments

Keep the plan concise and useful.
"""


    planning_response = llm.invoke(
        planning_prompt
    )

    research_plan = planning_response.content


    # ==========================================
    # STEP 2
    # ==========================================

    progress(
        progress_callback,
        2,
        "Gathering information and building FAISS knowledge index..."
    )


    wikipedia = get_wikipedia_information(
        topic
    )


    vector_store = create_faiss_store(
        topic,
        wikipedia
    )


    relevant_documents = (
        vector_store.similarity_search(
            research_plan,
            k=3
        )
    )


    retrieved_information = "\n\n".join(
        [
            doc.page_content
            for doc in relevant_documents
        ]
    )


    if not retrieved_information.strip():

        retrieved_information = (
            "No external information was retrieved. "
            "Use general knowledge carefully and "
            "avoid unsupported claims."
        )


    # ==========================================
    # STEP 3
    # ==========================================

    progress(
        progress_callback,
        3,
        "Analyzing and synthesizing the findings..."
    )


    synthesis_prompt = f"""
You are a senior research analyst.

Topic:
{topic}

Research plan:
{research_plan}

Retrieved information:
{retrieved_information}

Analyze and synthesize the available information.

Produce:

- Main findings
- Benefits
- Risks
- Challenges
- Real-world applications
- Practical implications
- Future outlook
- Evidence limitations

Do not invent statistics.
Do not invent sources.
Clearly identify uncertainty.
"""


    synthesis_response = llm.invoke(
        synthesis_prompt
    )

    synthesis = synthesis_response.content


    # ==========================================
    # STEP 4
    # ==========================================

    progress(
        progress_callback,
        4,
        "Generating the final research report..."
    )


    final_prompt = f"""
You are an expert research writer.

Create a professional research report.

Topic:
{topic}

Research plan:
{research_plan}

Retrieved information:
{retrieved_information}

Analysis:
{synthesis}

Use this structure:

# Research Report: {topic}

## Executive Summary

## Research Question

## Key Findings

## Detailed Analysis

## Benefits and Opportunities

## Risks and Challenges

## Real-World Applications

## Practical Implications

## Future Outlook

## Conclusion

## Sources and Verification Notes

Important rules:

- Do not fabricate citations.
- Do not fabricate URLs.
- Do not invent statistics.
- Clearly mention when information requires verification.
- Write for a non-technical reader.
- Keep the report well organized.
"""


    final_response = llm.invoke(
        final_prompt
    )

    report = final_response.content


    sources = []

    if wikipedia["url"]:

        sources.append({
            "title":
            wikipedia["title"],
            "url":
            wikipedia["url"]
        })


    return {

        "research_plan":
        research_plan,

        "retrieved_information":
        retrieved_information,

        "synthesis":
        synthesis,

        "report":
        report,

        "sources":
        sources
    }

Overwriting day30-ai-research-assistant/backend/research.py


In [8]:

%%writefile day30-ai-research-assistant/backend/evaluation.py

import json
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)

if not OPENAI_API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY is missing."
    )


evaluator = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)


def evaluate_report(
    topic,
    report
):

    prompt = f"""
You are a strict AI evaluation judge.

Evaluate the research report below.

TOPIC:
{topic}

REPORT:
{report}

Score each dimension from 0 to 10.

Dimensions:

1. Accuracy
2. Relevance
3. Completeness
4. Clarity
5. Structure

Overall must be the arithmetic average
of the five scores.

Return ONLY valid JSON.

Format:

{{
    "accuracy": 0,
    "relevance": 0,
    "completeness": 0,
    "clarity": 0,
    "structure": 0,
    "overall": 0,
    "feedback": "short feedback"
}}
"""


    response = evaluator.invoke(
        prompt
    )

    content = response.content.strip()


    # Remove accidental markdown fences

    if content.startswith("```"):

        content = (
            content
            .replace("```json", "")
            .replace("```", "")
            .strip()
        )


    result = json.loads(content)


    return result

Overwriting day30-ai-research-assistant/backend/evaluation.py


In [9]:

%%writefile day30-ai-research-assistant/backend/main.py

import asyncio
import json
import queue
import threading
import time

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field

from research import generate_research
from evaluation import evaluate_report


app = FastAPI(
    title="Day 30 AI Research Assistant",
    description=(
        "Full-stack AI Research Assistant "
        "with research, FAISS retrieval, "
        "and automatic evaluation."
    ),
    version="1.0.0"
)


# ==============================================
# CORS
# ==============================================

app.add_middleware(
    CORSMiddleware,

    allow_origins=["*"],

    allow_credentials=True,

    allow_methods=["*"],

    allow_headers=["*"]
)


# ==============================================
# Request Model
# ==============================================

class ResearchRequest(BaseModel):

    topic: str = Field(
        ...,
        min_length=3,
        max_length=500
    )


# ==============================================
# Root
# ==============================================

@app.get("/")
def root():

    return {
        "message":
        "Day 30 AI Research Assistant",

        "health":
        "/health",

        "research":
        "/research",

        "stream":
        "/research/stream",

        "docs":
        "/docs"
    }


# ==============================================
# Health
# ==============================================

@app.get("/health")
def health():

    return {
        "status": "healthy",

        "service":
        "AI Research Assistant",

        "day":
        30
    }


# ==============================================
# Worker
# ==============================================

def research_worker(
    topic,
    events
):

    start_time = time.time()


    def callback(
        step,
        message
    ):

        events.put({
            "type": "progress",
            "step": step,
            "message": message
        })


    try:

        result = generate_research(
            topic,
            progress_callback=callback
        )


        callback(
            4,
            "Evaluating the generated report..."
        )


        evaluation = evaluate_report(
            topic,
            result["report"]
        )


        events.put({

            "type":
            "complete",

            "topic":
            topic,

            "report":
            result["report"],

            "evaluation":
            evaluation,

            "sources":
            result["sources"],

            "processing_time":
            round(
                time.time() -
                start_time,
                2
            )
        })


    except Exception as error:

        events.put({

            "type":
            "error",

            "message":
            str(error)
        })


# ==============================================
# POST /research
# ==============================================

@app.post("/research")
def research(
    request: ResearchRequest
):

    topic = request.topic.strip()


    if not topic:

        raise HTTPException(
            status_code=400,
            detail="Topic cannot be empty."
        )


    events = queue.Queue()


    worker = threading.Thread(
        target=research_worker,
        args=(
            topic,
            events
        )
    )


    worker.start()


    while (
        worker.is_alive()
        or not events.empty()
    ):

        try:

            event = events.get(
                timeout=0.5
            )


            if event["type"] == "complete":

                return {
                    "success":
                    True,

                    **event
                }


            if event["type"] == "error":

                raise HTTPException(
                    status_code=500,
                    detail=event["message"]
                )


        except queue.Empty:

            continue


    raise HTTPException(
        status_code=500,
        detail="Research failed."
    )


# ==============================================
# GET /research/stream
# ==============================================

@app.get("/research/stream")
async def research_stream(
    topic: str
):

    topic = topic.strip()


    if len(topic) < 3:

        async def invalid():

            data = {
                "type":
                "error",

                "message":
                "Topic must contain at least 3 characters."
            }

            yield (
                f"data: "
                f"{json.dumps(data)}"
                f"\n\n"
            )


        return StreamingResponse(
            invalid(),
            media_type=
            "text/event-stream"
        )


    events = queue.Queue()


    worker = threading.Thread(
        target=research_worker,
        args=(
            topic,
            events
        )
    )


    worker.start()


    async def generator():

        while True:

            try:

                event = events.get_nowait()


                yield (
                    "data: "
                    +
                    json.dumps(event)
                    +
                    "\n\n"
                )


                if event["type"] in [
                    "complete",
                    "error"
                ]:

                    break


            except queue.Empty:

                yield ": heartbeat\n\n"

                await asyncio.sleep(
                    0.5
                )


    return StreamingResponse(

        generator(),

        media_type=
        "text/event-stream",

        headers={
            "Cache-Control":
            "no-cache",

            "Connection":
            "keep-alive",

            "X-Accel-Buffering":
            "no"
        }
    )

Overwriting day30-ai-research-assistant/backend/main.py


In [13]:

# ============================================
# FIX: research.py
# No OpenAI Python SDK required
# ============================================

import os

research_code = r'''
import os
import json
import requests

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY not found. Please set your API key first."
    )

OPENAI_URL = "https://api.openai.com/v1/chat/completions"


def call_openai(prompt):
    """
    Calls OpenAI API directly using requests.
    This avoids OpenAI SDK version conflicts.
    """

    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "gpt-4o-mini",
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an expert AI research assistant. "
                    "Give accurate, structured and useful research."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0.3,
        "max_tokens": 1800
    }

    response = requests.post(
        OPENAI_URL,
        headers=headers,
        json=payload,
        timeout=60
    )

    if response.status_code != 200:
        raise Exception(
            f"OpenAI API Error {response.status_code}: "
            f"{response.text}"
        )

    data = response.json()

    return data["choices"][0]["message"]["content"]


def generate_research(topic, progress_callback=None):
    """
    Four-step AI research workflow.

    Step 1: Research Planning
    Step 2: Key Findings
    Step 3: Analysis
    Step 4: Final Report
    """

    def progress(step, message):
        if progress_callback:
            progress_callback(step, message)

    # ----------------------------------------
    # STEP 1
    # ----------------------------------------

    progress(
        1,
        "Creating research plan..."
    )

    planning_prompt = f"""
Create a research plan for the following topic:

{topic}

Return:
1. Main research questions
2. Important areas to investigate
3. Key concepts
4. What a useful final report should contain
"""

    plan = call_openai(planning_prompt)

    # ----------------------------------------
    # STEP 2
    # ----------------------------------------

    progress(
        2,
        "Generating key findings..."
    )

    findings_prompt = f"""
Research the following topic using your general knowledge:

TOPIC:
{topic}

RESEARCH PLAN:
{plan}

Provide:
- Important facts
- Key findings
- Advantages
- Challenges
- Real-world applications
- Important examples

Clearly separate facts from assumptions.
"""

    findings = call_openai(findings_prompt)

    # ----------------------------------------
    # STEP 3
    # ----------------------------------------

    progress(
        3,
        "Analyzing the findings..."
    )

    analysis_prompt = f"""
Analyze the following research.

TOPIC:
{topic}

PLAN:
{plan}

FINDINGS:
{findings}

Provide:
- Main insights
- Trends
- Opportunities
- Risks
- Practical implications
- Balanced analysis
"""

    analysis = call_openai(analysis_prompt)

    # ----------------------------------------
    # STEP 4
    # ----------------------------------------

    progress(
        4,
        "Writing final research report..."
    )

    report_prompt = f"""
Create a professional research report.

TOPIC:
{topic}

RESEARCH PLAN:
{plan}

KEY FINDINGS:
{findings}

ANALYSIS:
{analysis}

Use this structure:

# Research Report

## Executive Summary

## Research Questions

## Key Findings

## Detailed Analysis

## Benefits / Opportunities

## Challenges / Risks

## Real-World Applications

## Conclusion

Write clearly so a non-technical reader can understand it.
Do not invent specific sources or citations.
"""

    report = call_openai(report_prompt)

    return {
        "topic": topic,
        "report": report,
        "sources": [
            "OpenAI model knowledge",
            "Research planning and analysis workflow"
        ],
        "workflow": {
            "step_1": "Research Planning",
            "step_2": "Key Findings",
            "step_3": "Analysis",
            "step_4": "Final Report"
        }
    }
'''

with open(
    "/content/day30-ai-research-assistant/backend/research.py",
    "w",
    encoding="utf-8"
) as f:
    f.write(research_code)

print("✅ research.py replaced successfully")

✅ research.py replaced successfully


In [22]:

# ============================================
# TEST IMPORTS
# ============================================

import sys

backend_path = "/content/day30-ai-research-assistant/backend"

if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

# Remove old cached module if it exists
if "research" in sys.modules:
    del sys.modules["research"]

from research import generate_research

print("✅ research.py imported successfully")
print("✅ generate_research is available")

✅ research.py imported successfully
✅ generate_research is available


In [19]:
!pip install -q python-dotenv requests

In [20]:

import os
from dotenv import load_dotenv

load_dotenv("/content/day30-ai-research-assistant/.env")

print(
    "API key loaded:",
    bool(os.getenv("OPENAI_API_KEY"))
)

API key loaded: True


In [ ]:

# ============================================================
# DAY 30 - GUARANTEED OFFLINE TEST
# OpenAI API ki zarurat nahi
# ============================================================

import sys
import os

BACKEND = "/content/day30-ai-research-assistant/backend"
os.makedirs(BACKEND, exist_ok=True)


def generate_research(topic, progress_callback=None):

    def progress(step, message):
        if progress_callback:
            progress_callback(step, message)

    # STEP 1
    progress(1, "Creating research plan...")

    plan = f"""
Research Plan for: {topic}

1. Understand the topic
2. Identify important concepts
3. Study benefits and applications
4. Analyze challenges and risks
5. Prepare a final conclusion
"""


    # STEP 2
    progress(2, "Generating key findings...")

    findings = f"""
Key Findings about {topic}:

• The topic has significant practical importance.
• Technology can improve productivity and accessibility.
• Proper implementation is required for reliable results.
• Human supervision remains important.
• Security, privacy and accuracy should be considered.
"""


    # STEP 3
    progress(3, "Analyzing research findings...")

    analysis = f"""
Analysis:

{topic} can provide meaningful benefits when implemented
with a clear strategy.

The major opportunities include better productivity,
automation, personalization and accessibility.

The major risks include inaccurate information,
privacy concerns, bias and overdependence on technology.

A balanced approach combining automation with human
oversight is recommended.
"""


    # STEP 4
    progress(4, "Writing final research report...")

    report = f"""
# Research Report

## Topic

{topic}

## Executive Summary

{topic} is an important area with significant potential.
It can improve productivity, accessibility and decision-making
when implemented responsibly.

## Research Questions

1. What is {topic}?
2. What are its major benefits?
3. What are its challenges?
4. Where can it be applied?
5. What is its future potential?

## Key Findings

{findings}

## Detailed Analysis

{analysis}

## Benefits / Opportunities

- Improved productivity
- Automation
- Better accessibility
- Personalized experiences
- Faster information processing

## Challenges / Risks

- Accuracy problems
- Privacy concerns
- Security risks
- Bias
- Overdependence on technology

## Real-World Applications

The technology can be applied in education, healthcare,
business, research, customer support and many other fields.

## Conclusion

{topic} has strong potential when used responsibly.
The best implementation combines technology with
human supervision and continuous evaluation.
"""


    return {
        "topic": topic,
        "report": report,
        "sources": [
            "Day 30 Research Workflow",
            "Offline Demo Research Engine"
        ],
        "workflow": {
            "step_1": "Research Planning",
            "step_2": "Key Findings",
            "step_3": "Analysis",
            "step_4": "Final Report"
        }
    }


print("✅ generate_research CREATED")
print("✅ OpenAI API NOT USED")
print("✅ No OpenAI credits required")

In [ ]:

import os

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

In [ ]:

import sys

backend_path = "/content/day30-ai-research-assistant/backend"

if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

# Remove old cached module
if "research" in sys.modules:
    del sys.modules["research"]

from research import generate_research

print("✅ generate_research imported successfully")

In [ ]:

result = generate_research(
    "Impact of Generative AI on Education",
    progress_callback=lambda step, message:
        print(f"STEP {step}: {message}")
)

print("\n" + "=" * 70)
print("RESEARCH REPORT")
print("=" * 70)

print(result["report"])

print("\n" + "=" * 70)
print("SOURCES")
print("=" * 70)

print(result["sources"])

print("\n" + "=" * 70)
print("WORKFLOW")
print("=" * 70)

print(result["workflow"])

In [ ]:

evaluation = evaluate_report(
    "Impact of Generative AI on Education",
    result["report"]
)

print(
    json.dumps(
        evaluation,
        indent=2
    )
)

In [ ]:

import sys
import nest_asyncio
import uvicorn

sys.path.insert(
    0,
    "/content/day30-ai-research-assistant/backend"
)

nest_asyncio.apply()

In [ ]:

config = uvicorn.Config(
    "main:app",
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

await server.serve()

In [ ]:

import requests

response = requests.get(
    "http://127.0.0.1:8000/health"
)

print("Status:", response.status_code)
print("Response:", response.json())

In [ ]:

import requests
import json

response = requests.post(

    "http://127.0.0.1:8000/research",

    json={
        "topic":
        "Impact of Generative AI on Education"
    },

    timeout=300
)

print(
    "HTTP Status:",
    response.status_code
)

data = response.json()

print(
    json.dumps(
        data,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:

import requests
import json

topic = "Future of Artificial Intelligence"

url = (
    "http://127.0.0.1:8000"
    "/research/stream"
)

with requests.get(
    url,
    params={"topic": topic},
    stream=True,
    timeout=300
) as response:

    print(
        "HTTP Status:",
        response.status_code
    )

    for line in response.iter_lines(
        decode_unicode=True
    ):

        if line:

            print(line)

In [ ]:

%%writefile day30-ai-research-assistant/evaluation/questions.json

[
  {
    "id": 1,
    "topic": "Impact of artificial intelligence on education"
  },
  {
    "id": 2,
    "topic": "Generative AI in healthcare"
  },
  {
    "id": 3,
    "topic": "Future of renewable energy"
  },
  {
    "id": 4,
    "topic": "Cybersecurity challenges in modern organizations"
  },
  {
    "id": 5,
    "topic": "Machine learning applications in finance"
  },
  {
    "id": 6,
    "topic": "Impact of social media on business"
  },
  {
    "id": 7,
    "topic": "Cloud computing trends"
  },
  {
    "id": 8,
    "topic": "Benefits and risks of autonomous vehicles"
  },
  {
    "id": 9,
    "topic": "AI agents and the future of software"
  },
  {
    "id": 10,
    "topic": "Importance of data privacy"
  },
  {
    "id": 11,
    "topic": "Applications of natural language processing"
  },
  {
    "id": 12,
    "topic": "Future of remote work"
  },
  {
    "id": 13,
    "topic": "Blockchain technology applications"
  },
  {
    "id": 14,
    "topic": "AI-assisted software development"
  },
  {
    "id": 15,
    "topic": "Digital transformation in businesses"
  },
  {
    "id": 16,
    "topic": "Role of big data in decision making"
  },
  {
    "id": 17,
    "topic": "Internet of Things applications"
  },
  {
    "id": 18,
    "topic": "Future of robotics"
  },
  {
    "id": 19,
    "topic": "AI ethics and responsible AI"
  },
  {
    "id": 20,
    "topic": "Future trends in artificial intelligence"
  }
]

In [ ]:

%%writefile day30-ai-research-assistant/evaluation/run_evaluation.py

import json
import os
import requests


BACKEND_URL = os.getenv(
    "BACKEND_URL",
    "http://127.0.0.1:8000"
)


def main():

    with open(
        "questions.json",
        "r",
        encoding="utf-8"
    ) as file:

        questions = json.load(file)


    results = []


    print(
        "\n================================"
    )

    print(
        "DAY 30 - 20 QUESTION EVALUATION"
    )

    print(
        "================================\n"
    )


    for question in questions:

        question_id = question["id"]

        topic = question["topic"]


        print(
            f"[{question_id}/20] "
            f"{topic}"
        )


        try:

            response = requests.post(

                f"{BACKEND_URL}/research",

                json={
                    "topic": topic
                },

                timeout=300
            )


            response.raise_for_status()


            data = response.json()


            evaluation = data[
                "evaluation"
            ]


            item = {

                "id":
                question_id,

                "topic":
                topic,

                "accuracy":
                evaluation["accuracy"],

                "relevance":
                evaluation["relevance"],

                "completeness":
                evaluation["completeness"],

                "clarity":
                evaluation["clarity"],

                "structure":
                evaluation["structure"],

                "overall":
                evaluation["overall"],

                "feedback":
                evaluation["feedback"]
            }


            results.append(item)


            print(
                "Overall:",
                evaluation["overall"],
                "/10"
            )


        except Exception as error:

            print(
                "ERROR:",
                error
            )


            results.append({

                "id":
                question_id,

                "topic":
                topic,

                "error":
                str(error)
            })


    valid = [
        item
        for item in results
        if "overall" in item
    ]


    metrics = [

        "accuracy",

        "relevance",

        "completeness",

        "clarity",

        "structure",

        "overall"

    ]


    baseline = {}


    if valid:

        for metric in metrics:

            values = [
                item[metric]
                for item in valid
            ]


            baseline[metric] = round(

                sum(values)
                /
                len(values),

                2
            )


    output = {

        "day":
        30,

        "total_questions":
        len(questions),

        "successful_questions":
        len(valid),

        "baseline":
        baseline,

        "results":
        results
    }


    with open(
        "results.json",
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            output,
            file,
            indent=2,
            ensure_ascii=False
        )


    print(
        "\n================================"
    )

    print(
        "OFFICIAL DAY 30 BASELINE"
    )

    print(
        "================================"
    )


    for metric, value in baseline.items():

        print(
            f"{metric}: {value}/10"
        )


    print(
        "\nSaved to results.json"
    )


if __name__ == "__main__":
    main()

In [ ]:

import subprocess
import os

os.chdir(
    "/content/day30-ai-research-assistant/evaluation"
)

result = subprocess.run(
    [
        "python",
        "run_evaluation.py"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("ERRORS:")
    print(result.stderr)

In [ ]:

import json

with open(
    "/content/day30-ai-research-assistant/evaluation/results.json",
    "r"
) as f:

    results = json.load(f)

print(
    json.dumps(
        results["baseline"],
        indent=2
    )
)

In [ ]:

%%writefile day30-ai-research-assistant/frontend/package.json

{
  "name": "day30-ai-research-assistant",
  "version": "1.0.0",
  "private": true,
  "scripts": {
    "dev": "next dev",
    "build": "next build",
    "start": "next start"
  },
  "dependencies": {
    "next": "14.2.35",
    "react": "^18.3.1",
    "react-dom": "^18.3.1"
  },
  "devDependencies": {
    "@types/node": "^20",
    "@types/react": "^18",
    "@types/react-dom": "^18",
    "typescript": "^5"
  }
}

In [ ]:

%%writefile day30-ai-research-assistant/frontend/app/layout.tsx

import "./globals.css";

export const metadata = {
  title: "AI Research Assistant",
  description:
    "Day 30 Full-Stack AI Research Assistant"
};

export default function RootLayout({
  children
}: {
  children: React.ReactNode;
}) {

  return (

    <html lang="en">

      <body>
        {children}
      </body>

    </html>
  );
}

In [ ]:

%%writefile day30-ai-research-assistant/frontend/app/globals.css

* {
  box-sizing: border-box;
}

body {
  margin: 0;
  font-family:
    Arial,
    Helvetica,
    sans-serif;

  background: #f4f7fb;
  color: #172033;
}

.page {
  width: min(
    1100px,
    92%
  );

  margin: auto;

  padding: 50px 0;
}

.hero {
  text-align: center;

  margin-bottom: 35px;
}

.hero h1 {
  font-size: 48px;

  margin: 15px 0;
}

.hero p {
  font-size: 18px;

  color: #667085;
}

.badge {
  display: inline-block;

  padding: 8px 14px;

  border-radius: 30px;

  background: #e8eefc;

  font-size: 13px;

  font-weight: bold;
}

.card {
  background: white;

  border-radius: 18px;

  padding: 28px;

  margin-bottom: 25px;

  box-shadow:
    0 8px 30px
    rgba(
      0,
      0,
      0,
      0.06
    );
}

.card h2 {
  margin-top: 0;
}

label {
  display: block;

  font-weight: bold;

  margin-bottom: 10px;
}

textarea {
  width: 100%;

  min-height: 140px;

  padding: 16px;

  border:
    1px solid
    #d0d5dd;

  border-radius: 12px;

  font-size: 16px;

  resize: vertical;
}

button {
  width: 100%;

  margin-top: 15px;

  padding: 15px;

  border: none;

  border-radius: 12px;

  background: #111827;

  color: white;

  font-size: 16px;

  font-weight: bold;

  cursor: pointer;
}

button:disabled {
  opacity: 0.5;

  cursor: not-allowed;
}

.error {
  margin-top: 15px;

  padding: 12px;

  border-radius: 10px;

  background: #fee4e2;

  color: #b42318;
}

.workflow {
  display: grid;

  gap: 15px;
}

.workflow-step {
  display: flex;

  align-items: center;

  gap: 15px;

  padding: 18px;

  border:
    1px solid
    #eaecf0;

  border-radius: 12px;

  transition:
    0.2s;
}

.workflow-step.completed {
  border-color:
    #12b76a;
}

.circle {
  width: 45px;

  height: 45px;

  min-width: 45px;

  border-radius: 50%;

  display: flex;

  align-items: center;

  justify-content: center;

  background: #eaecf0;

  font-weight: bold;
}

.workflow-step.completed
.circle {
  background:
    #12b76a;

  color: white;
}

.workflow-step p {
  margin: 5px 0;
}

.workflow-step small {
  color:
    #667085;
}

.activity {
  margin-top: 20px;

  padding: 16px;

  border-radius: 12px;

  background:
    #f8f9fc;
}

.activity p {
  margin: 8px 0;
}

.topic {
  padding: 12px;

  background:
    #f8f9fc;

  border-radius: 10px;

  margin-bottom: 20px;
}

.report {
  line-height: 1.8;
}

.report p {
  margin: 8px 0;
}

.sources {
  margin-top: 25px;

  padding-top: 20px;

  border-top:
    1px solid
    #eaecf0;
}

.sources a {
  display: block;

  margin: 8px 0;

  color: #475467;
}

.scores {
  display: grid;

  grid-template-columns:
    repeat(
      auto-fit,
      minmax(140px, 1fr)
    );

  gap: 15px;
}

.score {
  padding: 18px;

  border:
    1px solid
    #eaecf0;

  border-radius: 12px;

  display: flex;

  flex-direction: column;

  gap: 8px;
}

.score strong {
  font-size: 25px;
}

.score.highlight {
  background:
    #111827;

  color: white;
}

.feedback {
  margin-top: 20px;

  padding: 18px;

  background:
    #f8f9fc;

  border-radius: 12px;
}

@media (max-width: 600px) {

  .page {
    padding: 25px 0;
  }

  .hero h1 {
    font-size: 34px;
  }

  .card {
    padding: 20px;
  }
}

In [ ]:

%%writefile day30-ai-research-assistant/frontend/.env.local

NEXT_PUBLIC_API_URL=http://127.0.0.1:8000

In [ ]:

%%writefile day30-ai-research-assistant/.gitignore

# Python
__pycache__/
*.pyc
venv/
.env

# Node
node_modules/
.next/

# Environment
.env.local

# Jupyter
.ipynb_checkpoints/

# OS
.DS_Store

# IDE
.vscode/
.idea/

In [ ]:
cd backend

In [ ]:
python -m venv venv

In [ ]:
venv\Scripts\activate

In [ ]:
pip install -r requirements.txt

In [ ]:
OPENAI_API_KEY=your_api_key

In [ ]:
uvicorn main:app --reload

In [ ]:
http://127.0.0.1:8000/health

In [ ]:
http://127.0.0.1:8000/docs

In [ ]:

cd frontend
npm install

In [ ]:
NEXT_PUBLIC_API_URL=http://127.0.0.1:8000

In [ ]:
npm run dev

In [ ]:
http://localhost:3000